# Advanced 01 - Dialog, Failover, and Production Patterns

In this notebook you will:
- Understand SIP Dialog concept and Kamailio handling
- Learn failover routing patterns
- Understand RTP proxy (RTPEngine) integration
- Use performance-related variables and flags

---

## What is a SIP Dialog?

A Dialog is a **persistent relationship** between two SIP endpoints:

```
Alice          Kamailio          Bob
  |--- INVITE ---->|--- INVITE ---->|
  |<--- 100 ------|                |
  |<--- 180 ------|<--- 180 -------|
  |<--- 200 ------|<--- 200 -------|
  |--- ACK ------>|--- ACK ------->|
  |    [=== Call Active (Dialog) ===]   |
  |--- BYE ------>|--- BYE ------->|
  |<--- 200 ------|<--- 200 -------|
```

- Dialog identified by Call-ID + From-tag + To-tag
- `record_route()` ensures BYE/re-INVITE traverse the LB

## 1. In-Dialog vs Initial Request

This is the most fundamental branching logic in production.

In [ ]:
# Simulate initial INVITE
%%sip INVITE
From: <sip:1001@example.com>;tag=a1b2c3
To: <sip:1002@example.com>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# Initial request: no To tag
if (has_totag()) {
    xlog("In-dialog: relay to existing dialog target");
    t_relay();
} else {
    xlog("Initial request: $(ru{uri.user}) is calling $(tu{uri.user})");
    
    if (is_method("INVITE")) {
        record_route();
        lookup("location");
        t_relay();
    }
    if (is_method("REGISTER")) {
        save("location");
    }
}

## 2. Failover Pattern

When one node doesn't respond, try the next one:

```
INVITE → Node1 (no response) → Node2 (success)
```

In Kamailio, use `ds_next_dst()` for failover.

In [ ]:
$var(retry_count) = 0;

# Select first destination
ds_select_dst("1", "4");
xlog("Try 1: selected destination");

# On failure, try next
$var(retry_count) = 1;
ds_next_dst();
xlog("Try 2: next destination (failover)");

# Mark first destination as probing
ds_mark_dst("p");
xlog("Marked first destination as probing");

## 3. RTP Proxy (RTPEngine) Integration

Media (audio/video) takes a **separate path** from SIP signaling:

```
Alice ──[SIP signal]──> Kamailio ──[SIP signal]──> Bob
  \                                                  /
   ────────[RTP media]──> RTPEngine <────────────
```

`rtpengine_manage()` automatically handles offer/answer/delete based on message type.

In [ ]:
%%sip INVITE
From: <sip:1001@192.168.1.100>;tag=rtp1
To: <sip:1002@192.168.1.200>
Contact: <sip:1001@192.168.1.100:5060>

In [ ]:
# Typical RTPEngine processing sequence
xlog("Step 1: Route to destination");
ds_select_dst("1", "4");

xlog("Step 2: Add Record-Route");
record_route();

xlog("Step 3: Engage RTPEngine");
rtpengine_manage("replace-origin replace-session-connection");

xlog("Step 4: Relay");
t_relay();

## 4. Flags and State Management

Kamailio uses flags to track per-request state.

| Function | Description |
|----------|-------------|
| `setflag(n)` | Set flag n |
| `resetflag(n)` | Clear flag n |
| `isflagset(n)` | Check flag n |

In [ ]:
setflag(1);  # e.g., "rtp_engine_enabled"

if (isflagset(1)) {
    xlog("RTP engine is enabled for this call");
    rtpengine_manage("replace-origin");
} else {
    xlog("Direct media — no RTP proxy");
}

resetflag(1);

## 5. Header Manipulation

Adding/removing SIP headers is very common in production.

In [ ]:
# Add header
append_hf("X-Call-Source: external");
xlog("Added custom header");

# Check presence
is_present_hf("X-Call-Source");

# Remove header
remove_hf("X-Call-Source");
xlog("Removed custom header");

## 6. Execution Tracing (v0.2)

The `%%trace` magic traces code execution paths in detail.

In [ ]:
%%trace
%%sip INVITE
From: <sip:1001@example.com>;tag=trace1
To: <sip:1002@example.com>

In [ ]:
%%trace
if (is_method("INVITE")) {
    xlog("INVITE received");
    record_route();
    ds_select_dst("1", "4");
    t_relay();
} else {
    xlog("Not INVITE");
    send_reply(405, "Method Not Allowed");
}

## 7. Production Routing Template

The basic `request_route` structure used in production:

In [ ]:
# Standard request_route pattern

# 1. Basic validation
if (!is_method("INVITE|REGISTER|BYE|ACK|CANCEL|OPTIONS")) {
    send_reply(405, "Method Not Allowed");
    exit;
}
xlog("[1] Method validated: $rm");

# 2. In-dialog handling
if (has_totag()) {
    xlog("[2] In-dialog request");
    route(RELAY);
}

# 3. REGISTER
if (is_method("REGISTER")) {
    xlog("[3] Registration");
    save("location");
    exit;
}

# 4. INVITE routing
if (is_method("INVITE")) {
    xlog("[4] INVITE routing");
    record_route();
    lookup("location");
    rtpengine_manage("replace-origin");
    t_relay();
}

In [ ]:
%%vars

In [ ]:
%%diff

---

### Production Pattern Summary

| Pattern | Key Function | Description |
|---------|-------------|-------------|
| Dialog branching | `has_totag()` | Initial vs in-dialog |
| Failover | `ds_select_dst()` + `ds_next_dst()` | Try next on failure |
| RTP proxy | `rtpengine_manage()` | Media relay |
| Flags | `setflag()` / `isflagset()` | Per-request state |
| Headers | `append_hf()` / `remove_hf()` | Add/remove SIP headers |
| Tracing | `%%trace` | Code path visualization |
| Diff | `%%diff` | Pre/post execution comparison |

---

### Further Learning

- [Kamailio Wiki](https://www.kamailio.org/wiki/)
- [Kamailio cfg Cookbook](https://www.kamailio.org/wiki/cookbooks/5.5.x/core)
- [RFC 3261 - SIP](https://www.rfc-editor.org/rfc/rfc3261)

You've completed the entire curriculum!